# 03 — From source-scoped evidence to a grounded answer

**10–15 minute lab.** Build one typed retrieval result, analyze it independently, then let the
real `GenerationPipeline` bind provenance and synthesize only from relevant facts.

**Flow:** retrieved source → relevant facts → pipeline-bound lineage → grounded answer.


In [ ]:
from raglab import Citation, ProvenanceStatus
from raglab.errors import GenerationError
from raglab.generation import (
    GenerationConfig,
    GenerationPipeline,
    GenerationRequest,
    ModelInvocation,
)
from raglab.retrieval import (
    CollectionMetadata,
    RankingTrace,
    RetrievalRequest,
    RetrievalResponse,
    RetrievalResult,
)


class StaticRetrieval:
    def __init__(self, results):
        self.results = results

    def collection_metadata(self, collection):
        return CollectionMetadata(collection, "hermetic-embedding", 2)

    def retrieve(self, request):
        return RetrievalResponse(
            query=request.query,
            rewritten_query="retrieval-only wording",
            query_variants=(request.query,),
            filters=request.filters,
            results=self.results,
        )


class EvidenceModel:
    def __init__(self, facts, answer):
        self.facts = facts
        self.answer = answer
        self.calls = []

    def generate(self, prompt, *, system, schema, config):
        self.calls.append((prompt, system, schema))
        if "analyze exactly one" in system:
            return ModelInvocation({"facts": self.facts}, 80, 18)
        return ModelInvocation({"answer": self.answer}, 60, 14)


## Checkpoint 1 — Objective: create grounded evidence

**Run:** construct the same public retrieval contract used by the production pipeline.


In [ ]:
citation = Citation(
    source_uri="memory://aster-manual",
    source_name="aster-manual.md",
    title="Aster Greenhouse Controller",
    heading_path=("Fault E17",),
    start_page=None,
    end_page=None,
    start_line=42,
    end_line=45,
    provenance_status=ProvenanceStatus.COMPLETE,
)
evidence = RetrievalResult(
    id="e17-evidence",
    document_id="aster-manual",
    content="Fault E17 means irrigation flow stayed below the safe threshold.",
    citation=citation,
    matched_chunk_ids=("chunk-7",),
    first_chunk_index=7,
    last_chunk_index=7,
    trace=RankingTrace(1, 1, 0.05, 9.2, 0.032, None, None),
)
print(evidence.content)
print(
    f"Source: {evidence.citation.source_name}, "
    f"lines {evidence.citation.start_line}-{evidence.citation.end_line}"
)


### What to observe

Expect full evidence text plus traceable source lines. The model does not receive an anonymous
snippet; it receives evidence that can support a citation.

### Conclusion

Grounding starts before generation: retrieval must preserve both content and provenance.


## Checkpoint 2 — Objective: generate from pipeline-bound facts

**Run:** use a deterministic model adapter while keeping RAGLab's real source analysis,
deduplication, synthesis, and lineage resolution.


In [ ]:
request = GenerationRequest(
    retrieval=RetrievalRequest(
        "What does fault E17 mean?",
        collection="lab",
        history=("We are diagnosing the greenhouse controller.",),
    ),
    config=GenerationConfig(minimum_sources=1),
)
model = EvidenceModel(
    ["Fault E17 means irrigation flow stayed below the safe threshold."],
    "E17 reports irrigation flow below the safe threshold.",
)
pipeline = GenerationPipeline(
    StaticRetrieval((evidence,)),
    model,
    embedding_model="hermetic-embedding",
    embedding_dimension=2,
)
response = pipeline.generate(request)
print(response.answer)
print("Pipeline-derived source IDs:", response.source_ids)
print("Model calls:", response.metrics.model_calls)


### What to observe

Expect two model calls: one source analysis and one synthesis. The model schemas never ask for a
source ID; the pipeline attaches `S1` to the extracted fact and derives the public lineage.
The prompts use the original history and question, not the rewritten retrieval query.

### Conclusion

Claim lineage is established before synthesis. Attribution is pipeline state, not model output.


## Checkpoint 3 — Objective: abstain before synthesis when evidence is irrelevant

**Run:** make the source analysis emit no relevant facts and observe canonical abstention.


In [ ]:
unsupported_model = EvidenceModel([], "This synthesis must not run.")
unsupported_pipeline = GenerationPipeline(
    StaticRetrieval((evidence,)),
    unsupported_model,
    embedding_model="hermetic-embedding",
    embedding_dimension=2,
)
unsupported = unsupported_pipeline.generate(
    GenerationRequest(
        RetrievalRequest("What is the controller password?", collection="lab"),
        GenerationConfig(minimum_sources=1),
    )
)
print(unsupported.answer)
print("Abstained:", unsupported.abstained)
print("Model calls:", unsupported.metrics.model_calls)
assert unsupported.abstained
assert unsupported.metrics.model_calls == 1


### What to observe

Expect the canonical no-relevant-facts response, no sources, and one model call. Synthesis never
runs because the pipeline—not the model—decides that the evidence set is empty.

### Conclusion

Source-scoped filtering prevents irrelevant retrieved text from becoming an answer and makes
unsupported questions abstain deterministically.

## Optional appendix — live local generation

Use `raglab-generate` with an indexed collection to explore Ollama and source shortfall. An
oversized individual source fails explicitly; RAGLab does not silently truncate evidence.
